In [1]:
import sys
print(sys.executable)
%pip install "pyspark>=4.0"

c:\Users\Salah\AppData\Local\Programs\Python\Python311\python.exe
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pyspark
print(pyspark.__version__)

4.3.0


In [3]:
!java -version

openjdk version "25.0.5" 2026-10-20 LTS
OpenJDK Runtime Environment Temurin-25.0.5+8 (build 25.0.5+8-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.5+8 (build 25.0.5+8-LTS, mixed mode, sharing)


In [4]:
import pyspark
print(pyspark.__version__)

4.3.0


In [5]:
import os, csv, json, random
from datetime import date, timedelta

random.seed(42)
os.makedirs("data_tp2", exist_ok=True)

# ---------- produits.csv ----------
catalogue = {
    "Informatique": ["Clavier", "Souris", "Ecran", "Casque", "Webcam"],
    "Maison":       ["Lampe", "Chaise", "Bureau", "Etagere"],
    "Sport":        ["Ballon", "Raquette", "Tapis de yoga", "Haltères"],
}
produits_l, pid = [], 1
for cat, noms in catalogue.items():
    for n in noms:
        produits_l.append([pid, n, cat, random.choice([49, 79, 99, 149, 249, 399, 599])])
        pid += 1
produits_l[3][3] = "N/A"          # prix invalide (produit 4)
with open("data_tp2/produits.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_produit", "nom", "categorie", "prix"])
    w.writerows(produits_l)

# ---------- clients.json (un objet JSON par ligne) ----------
villes = ["Casablanca", "Rabat", "Fès", "Marrakech", "Tanger", "Agadir"]
prenoms = ["Sara", "Youssef", "Imane", "Omar", "Salma",
           "Hamza", "Khadija", "Anas", "Meryem", "Ayoub"]
familles = ["Alaoui", "Benani", "Cherkaoui", "Darif", "El Idrissi", "Fassi", "Tazi"]
with open("data_tp2/clients.json", "w", encoding="utf-8") as f:
    for i in range(1, 61):
        ville = random.choice(villes)
        if i % 7 == 0:
            ville = " " + ville.upper() + " "     # ville mal saisie
        c = {"id_client": i,
             "nom": random.choice(prenoms) + " " + random.choice(familles),
             "ville": ville,
             "date_inscription": (date(2024, 1, 1)
                                  + timedelta(days=random.randint(0, 364))).isoformat()}
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# ---------- commandes.csv ----------
statuts = ["livree", "livree", "livree", "annulee", "en_cours", "Livree", "LIVREE "]
cmd = []
for i in range(1, 3001):
    d = date(2025, 1, 1) + timedelta(days=random.randint(0, 364))
    cmd.append([i, random.randint(1, 54), random.randint(1, 13),
                random.randint(1, 5), d.isoformat(), random.choice(statuts)])
for k in range(0, 3000, 150): cmd[k][3] = "abc"     # quantité invalide
for k in range(5, 3000, 200): cmd[k][3] = -2        # quantité négative
for k in range(10, 3000, 300): cmd[k][4] = ""       # date manquante
for k in range(20, 3000, 500): cmd[k][1] = ""       # client manquant
cmd += random.sample(cmd, 40)                       # doublons exacts
with open("data_tp2/commandes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_commande", "id_client", "id_produit",
                "quantite", "date_commande", "statut"])
    w.writerows(cmd)

print(sorted(os.listdir("data_tp2")))

['clients.json', 'commandes.csv', 'produits.csv']


In [6]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TP2 - Ventes")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", 8)
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Version Spark :", spark.version)
print("Mode ANSI     :", spark.conf.get("spark.sql.ansi.enabled"))
print("Partitions    :", spark.conf.get("spark.sql.shuffle.partitions"))

Version Spark : 4.3.0
Mode ANSI     : true
Partitions    : 8


In [7]:
# 1) Schémas explicites : toutes les colonnes en STRING
schema_produits  = "id_produit STRING, nom STRING, categorie STRING, prix STRING"
schema_commandes = ("id_commande STRING, id_client STRING, id_produit STRING, "
                    "quantite STRING, date_commande STRING, statut STRING")

# 2) Lecture des CSV avec header + schéma
produits_raw = (spark.read
                .option("header", True)
                .schema(schema_produits)
                .csv("data_tp2/produits.csv"))

commandes_raw = (spark.read
                 .option("header", True)
                 .schema(schema_commandes)
                 .csv("data_tp2/commandes.csv"))

# 3) Lecture du JSON (un objet par ligne)
clients_raw = spark.read.json("data_tp2/clients.json")

# 4) Schéma, 5 premières lignes et nombre de lignes
for nom, df in [("produits_raw", produits_raw),
                ("clients_raw", clients_raw),
                ("commandes_raw", commandes_raw)]:
    print(f"===== {nom} : {df.count()} lignes =====")
    df.printSchema()
    df.show(5, truncate=False)

===== produits_raw : 13 lignes =====
root
 |-- id_produit: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: string (nullable = true)

+----------+-------+------------+----+
|id_produit|nom    |categorie   |prix|
+----------+-------+------------+----+
|1         |Clavier|Informatique|399 |
|2         |Souris |Informatique|49  |
|3         |Ecran  |Informatique|49  |
|4         |Casque |Informatique|N/A |
|5         |Webcam |Informatique|99  |
+----------+-------+------------+----+
only showing top 5 rows
===== clients_raw : 60 lignes =====
root
 |-- date_inscription: string (nullable = true)
 |-- id_client: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)

+----------------+---------+---------------+----------+
|date_inscription|id_client|nom            |ville     |
+----------------+---------+---------------+----------+
|2024-01-17      |1        |Ayoub Darif    |Casablanca|
|20

In [8]:
manquants = commandes_raw.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1)
    ).alias(c)
    for c in commandes_raw.columns
])
manquants.show()

+-----------+---------+----------+--------+-------------+------+
|id_commande|id_client|id_produit|quantite|date_commande|statut|
+-----------+---------+----------+--------+-------------+------+
|          0|        7|         0|       0|           10|     0|
+-----------+---------+----------+--------+-------------+------+



In [9]:
n_total = commandes_raw.count()
n_dedup = commandes_raw.dropDuplicates().count()

print(f"Lignes totales    : {n_total}")
print(f"Lignes distinctes : {n_dedup}")
print(f"Doublons exacts   : {n_total - n_dedup}")

Lignes totales    : 3040
Lignes distinctes : 3000
Doublons exacts   : 40


In [10]:
(commandes_raw
 .groupBy("statut").count()
 .withColumn("statut", F.concat(F.lit("'"), F.col("statut"), F.lit("'")))
 .orderBy(F.desc("count"))
 .show(truncate=False))

print("Valeurs distinctes de statut :",
      commandes_raw.select("statut").distinct().count())

+----------+-----+
|statut    |count|
+----------+-----+
|'livree'  |1260 |
|'annulee' |476  |
|'Livree'  |465  |
|'LIVREE ' |426  |
|'en_cours'|413  |
+----------+-----+

Valeurs distinctes de statut : 5


In [11]:
# Conversion "sûre" de la quantité en entier
cmd_q = commandes_raw.withColumn("quantite_int", F.col("quantite").try_cast("int"))

# Lignes invalides : conversion impossible (null) OU quantité <= 0
quantites_invalides = cmd_q.filter(
    F.col("quantite_int").isNull() | (F.col("quantite_int") <= 0)
)

print("Lignes concernées :", quantites_invalides.count())

# Détail par valeur fautive
quantites_invalides.groupBy("quantite").count().show()

# 5 exemples
quantites_invalides.show(5)

Lignes concernées : 35
+--------+-----+
|quantite|count|
+--------+-----+
|     abc|   20|
|      -2|   15|
+--------+-----+

+-----------+---------+----------+--------+-------------+-------+------------+
|id_commande|id_client|id_produit|quantite|date_commande| statut|quantite_int|
+-----------+---------+----------+--------+-------------+-------+------------+
|          1|       51|         8|     abc|   2025-05-05| livree|        NULL|
|          6|       35|         8|      -2|   2025-04-08|annulee|          -2|
|        151|       25|         8|     abc|   2025-09-25| livree|        NULL|
|        206|       44|         5|      -2|   2025-11-20| Livree|          -2|
|        301|       43|         5|     abc|   2025-09-27| livree|        NULL|
+-----------+---------+----------+--------+-------------+-------+------------+
only showing top 5 rows


In [12]:
# Produits dont le prix n'est pas convertible en nombre
produits_raw.filter(F.col("prix").try_cast("double").isNull()).show()

# Pour comparaison : toutes les valeurs de prix
produits_raw.select("prix").distinct().orderBy("prix").show()

+----------+------+------------+----+
|id_produit|   nom|   categorie|prix|
+----------+------+------------+----+
|         4|Casque|Informatique| N/A|
+----------+------+------------+----+

+----+
|prix|
+----+
| 249|
| 399|
|  49|
|  79|
|  99|
| N/A|
+----+



In [13]:
avant = commandes_raw.count()

commandes = (commandes_raw
    # R1 : doublons exacts
    .dropDuplicates()

    # R2 : typage sûr
    .withColumn("id_commande",   F.col("id_commande").try_cast("int"))
    .withColumn("id_client",     F.col("id_client").try_cast("int"))
    .withColumn("id_produit",    F.col("id_produit").try_cast("int"))
    .withColumn("quantite",      F.col("quantite").try_cast("int"))
    .withColumn("date_commande", F.col("date_commande").try_cast("date"))

    # R3 : normalisation du statut
    .withColumn("statut", F.lower(F.trim(F.col("statut"))))

    # R4 : filtres (après le typage)
    .filter(F.col("quantite").isNotNull() & (F.col("quantite") > 0))
    .filter(F.col("date_commande").isNotNull())
    .filter(F.col("id_client").isNotNull())
)

apres = commandes.count()
print(f"Avant nettoyage : {avant} lignes")
print(f"Après nettoyage : {apres} lignes ({avant - apres} supprimées)")

commandes.printSchema()
commandes.groupBy("statut").count().orderBy(F.desc("count")).show()

Avant nettoyage : 3040 lignes
Après nettoyage : 2949 lignes (91 supprimées)
root
 |-- id_commande: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_produit: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- statut: string (nullable = true)

+--------+-----+
|  statut|count|
+--------+-----+
|  livree| 2089|
| annulee|  463|
|en_cours|  397|
+--------+-----+



In [14]:
# Fenêtre : un "groupe" par catégorie, sans regrouper les lignes
w = Window.partitionBy("categorie")

produits = (produits_raw
    # R2 : typage
    .withColumn("id_produit", F.col("id_produit").try_cast("int"))
    .withColumn("prix",       F.col("prix").try_cast("double"))
    # R5 : prix null -> moyenne de la catégorie
    .withColumn("prix", F.round(
        F.coalesce(F.col("prix"), F.avg("prix").over(w)), 2))
    .orderBy("id_produit")
)

produits.printSchema()
produits.show()

root
 |-- id_produit: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: double (nullable = true)

+----------+-------------+------------+-----+
|id_produit|          nom|   categorie| prix|
+----------+-------------+------------+-----+
|         1|      Clavier|Informatique|399.0|
|         2|       Souris|Informatique| 49.0|
|         3|        Ecran|Informatique| 49.0|
|         4|       Casque|Informatique|149.0|
|         5|       Webcam|Informatique| 99.0|
|         6|        Lampe|      Maison| 79.0|
|         7|       Chaise|      Maison| 79.0|
|         8|       Bureau|      Maison| 79.0|
|         9|      Etagere|      Maison|399.0|
|        10|       Ballon|       Sport| 49.0|
|        11|     Raquette|       Sport|399.0|
|        12|Tapis de yoga|       Sport|399.0|
|        13|     Haltères|       Sport|249.0|
+----------+-------------+------------+-----+



In [15]:
# Moyenne Informatique calculée à la main, sans le Casque
(produits_raw
 .filter((F.col("categorie") == "Informatique") & (F.col("id_produit") != "4"))
 .agg(F.avg(F.col("prix").cast("double")).alias("moyenne_informatique"))
 .show())

+--------------------+
|moyenne_informatique|
+--------------------+
|               149.0|
+--------------------+



In [16]:
(clients_raw
 .select(F.concat(F.lit("'"), F.col("ville"), F.lit("'")).alias("ville"))
 .distinct().orderBy("ville")
 .show(truncate=False))

+--------------+
|ville         |
+--------------+
|' CASABLANCA '|
|' MARRAKECH ' |
|' RABAT '     |
|' TANGER '    |
|'Agadir'      |
|'Casablanca'  |
|'Fès'         |
|'Marrakech'   |
|'Rabat'       |
|'Tanger'      |
+--------------+



In [17]:
clients = (clients_raw
    # R2 : typage
    .withColumn("id_client",        F.col("id_client").cast("int"))
    .withColumn("date_inscription", F.col("date_inscription").try_cast("date"))
    # R3 : normalisation de la ville
    .withColumn("ville", F.initcap(F.trim(F.col("ville"))))
)

clients.printSchema()

villes = clients.select("ville").distinct().orderBy("ville")
print("Nombre de villes distinctes :", villes.count())
villes.show()

root
 |-- date_inscription: date (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)

Nombre de villes distinctes : 6
+----------+
|     ville|
+----------+
|    Agadir|
|Casablanca|
|       Fès|
| Marrakech|
|     Rabat|
|    Tanger|
+----------+



In [18]:
produits.createOrReplaceTempView("produits")
clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")

spark.sql("SHOW VIEWS").show()

+---------+---------+-----------+
|namespace| viewName|isTemporary|
+---------+---------+-----------+
|         |  clients|       true|
|         |commandes|       true|
|         | produits|       true|
+---------+---------+-----------+



In [19]:
spark.sql("""
    SELECT ROUND(SUM(c.quantite * p.prix), 2) AS ca_total,
           COUNT(*)                           AS nb_commandes_livrees
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
""").show()

+---------+--------------------+
| ca_total|nb_commandes_livrees|
+---------+--------------------+
|1215123.0|                2089|
+---------+--------------------+



In [20]:
spark.sql("""
    SELECT p.categorie,
           ROUND(SUM(c.quantite * p.prix), 2)            AS ca,
           COUNT(*)                                      AS nb_commandes,
           ROUND(SUM(c.quantite * p.prix) / COUNT(*), 2) AS panier_moyen
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.categorie
    ORDER BY ca DESC
""").show()

+------------+--------+------------+------------+
|   categorie|      ca|nb_commandes|panier_moyen|
+------------+--------+------------+------------+
|       Sport|574687.0|         685|      838.96|
|Informatique|354311.0|         800|      442.89|
|      Maison|286125.0|         604|      473.72|
+------------+--------+------------+------------+



Sport est la catégorie la plus rentable (574 687, soit environ 47 % du CA total), alors qu’elle a moins de commandes qu’Informatique (685 contre 800). C’est son panier moyen élevé (838.96) qui fait la différence : trois de ses produits sont chers (Raquette et Tapis de yoga à 399, Haltères à 249). À l’inverse, Informatique a le plus de commandes mais le panier moyen le plus faible (442.89), car la Souris et l’Écran ne coûtent que 49. Le volume de commandes ne suffit donc pas à expliquer le CA : il faut regarder aussi le prix des produits.

In [21]:
ca_mensuel = spark.sql("""
    SELECT date_format(c.date_commande, 'yyyy-MM') AS mois,
           ROUND(SUM(c.quantite * p.prix), 2)      AS ca,
           COUNT(*)                                AS nb_commandes
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY date_format(c.date_commande, 'yyyy-MM')
    ORDER BY mois
""")
ca_mensuel.show()

# Mois au CA le plus élevé
meilleur = ca_mensuel.orderBy(F.desc("ca")).first()
print(f"Mois au CA le plus élevé : {meilleur['mois']} avec un CA de {meilleur['ca']:,.0f}")

+-------+--------+------------+
|   mois|      ca|nb_commandes|
+-------+--------+------------+
|2025-01|118017.0|         183|
|2025-02| 98790.0|         163|
|2025-03|109451.0|         189|
|2025-04| 81453.0|         145|
|2025-05| 86787.0|         158|
|2025-06|114252.0|         186|
|2025-07|106081.0|         193|
|2025-08|114002.0|         186|
|2025-09| 92077.0|         155|
|2025-10| 90417.0|         164|
|2025-11| 99390.0|         173|
|2025-12|104406.0|         194|
+-------+--------+------------+

Mois au CA le plus élevé : 2025-01 avec un CA de 118,017


In [22]:
spark.sql("""
    SELECT p.nom,
           p.categorie,
           SUM(c.quantite)                    AS quantite_totale,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.nom, p.categorie
    ORDER BY ca DESC
    LIMIT 5
""").show()

+-------------+------------+---------------+--------+
|          nom|   categorie|quantite_totale|      ca|
+-------------+------------+---------------+--------+
|     Raquette|       Sport|            552|220248.0|
|Tapis de yoga|       Sport|            517|206283.0|
|      Clavier|Informatique|            491|195909.0|
|      Etagere|      Maison|            451|179949.0|
|     Haltères|       Sport|            485|120765.0|
+-------------+------------+---------------+--------+



In [23]:
(commandes.filter(F.col("statut") == "livree")
 .join(produits, "id_produit")
 .agg(F.round(F.sum(F.col("quantite") * F.col("prix")), 2).alias("ca_total"),
      F.count("*").alias("nb_commandes_livrees"))
 .show())

+---------+--------------------+
| ca_total|nb_commandes_livrees|
+---------+--------------------+
|1215123.0|                2089|
+---------+--------------------+



Ce TP montre qu’il faut toujours auditer avant d’analyser. Le chargement en STRING combiné à try_cast permet de traiter des données sales sans faire planter Spark en mode ANSI. Sans nettoyage, le CA aurait été faux : 891 commandes livrées auraient été ignorées à cause de la casse du statut, et les commandes du Casque auraient eu un prix null. Les vues temporaires permettent ensuite d’écrire les analyses en SQL, avec le même résultat qu’en API DataFrame.

In [24]:
spark.stop()